In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install librosa torch --quiet

import librosa
import librosa.display
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import torch
import torch.nn as nn


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
DATA_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
GENRES_PATH = f"{DATA_PATH}/genres_stems"
ESC_PATH = f"{DATA_PATH}/ESC-50-master/audio"


In [ ]:
genre = "rock"
song_folder = os.listdir(f"{GENRES_PATH}/{genre}")[0]
file_path = f"{GENRES_PATH}/{genre}/{song_folder}/vocals.wav"
audio, sr = librosa.load(file_path, sr=None)
print(sr, len(audio)/sr)


In [ ]:
plt.figure(figsize=(12,4))
librosa.display.waveshow(audio, sr=sr)
plt.show()


In [ ]:
spec = librosa.feature.melspectrogram(y=audio, sr=sr)
spec_db = librosa.power_to_db(spec, ref=np.max)

plt.figure(figsize=(12,4))
librosa.display.specshow(spec_db, sr=sr, x_axis='time', y_axis='mel')
plt.colorbar()
plt.show()


In [ ]:
genre_counts = {}

for genre in os.listdir(GENRES_PATH):
    genre_counts[genre] = len(os.listdir(f"{GENRES_PATH}/{genre}"))

pd.Series(genre_counts).sort_values().plot(kind="bar", figsize=(10,4))
plt.title("Number of Songs per Genre")
plt.show()


In [ ]:
lengths = []

for genre in os.listdir(GENRES_PATH):
    songs = os.listdir(f"{GENRES_PATH}/{genre}")
    
    for song in songs:
        path = f"{GENRES_PATH}/{genre}/{song}/vocals.wav"
        if os.path.exists(path):
            audio, sr = librosa.load(path, sr=None)
            lengths.append(len(audio)/sr)

plt.hist(lengths, bins=20)
plt.title("Audio Length Distribution")
plt.xlabel("Seconds")
plt.show()

print("Mean length:", np.mean(lengths))


In [ ]:
sample_rates = []

for genre in os.listdir(GENRES_PATH):
    songs = os.listdir(f"{GENRES_PATH}/{genre}")[:10]
    for song in songs:
        path = f"{GENRES_PATH}/{genre}/{song}/vocals.wav"
        if os.path.exists(path):
            _, sr = librosa.load(path, sr=None)
            sample_rates.append(sr)

pd.Series(sample_rates).value_counts()


In [ ]:
def plot_genre_mfcc(genre):
    song = os.listdir(f"{GENRES_PATH}/{genre}")[0]
    path = f"{GENRES_PATH}/{genre}/{song}/vocals.wav"
    audio, sr = librosa.load(path, sr=None)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
    
    plt.figure(figsize=(8,3))
    librosa.display.specshow(mfcc, x_axis='time')
    plt.title(genre)
    plt.show()

plot_genre_mfcc("rock")
plot_genre_mfcc("classical")
plot_genre_mfcc("hiphop")


In [ ]:
mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)

plt.figure(figsize=(10,4))
librosa.display.specshow(mfcc, x_axis='time')
plt.colorbar()
plt.title("MFCC Example")
plt.show()


In [ ]:
def silence_ratio(file_path):
    y, sr = librosa.load(file_path, sr=None)
    intervals = librosa.effects.split(y, top_db=20)
    voiced = sum((end-start) for start,end in intervals)
    return 1 - voiced/len(y)

ratios = []
for genre in os.listdir(GENRES_PATH):
    songs = os.listdir(f"{GENRES_PATH}/{genre}")[:20]
    for song in songs:
        path = f"{GENRES_PATH}/{genre}/{song}/vocals.wav"
        if os.path.exists(path):
            ratios.append(silence_ratio(path))

plt.hist(ratios, bins=20)
plt.title("Silence Ratio Distribution")
plt.show()


In [ ]:
def extract_features(file_path):
    audio, sr = librosa.load(file_path, sr=None)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=20)
    return np.mean(mfcc.T, axis=0)


In [ ]:
genres = os.listdir(GENRES_PATH)

X = []
y = []

for genre in genres:
    songs = os.listdir(f"{GENRES_PATH}/{genre}")
    for song in tqdm(songs):
        file_path = f"{GENRES_PATH}/{genre}/{song}/vocals.wav"
        features = extract_features(file_path)
        X.append(features)
        y.append(genre)

X = np.array(X)
y = np.array(y)


In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X, y_encoded)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
X_val = torch.tensor(X_val, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.long).to(device)
y_val = torch.tensor(y_val, dtype=torch.long).to(device)


In [ ]:
class GenreNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(20,128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,10)
        )
    def forward(self,x):
        return self.model(x)

model = GenreNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
for epoch in range(25):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_preds = model(X_val)
        val_loss = criterion(val_preds, y_val)

    print(epoch+1, loss.item(), val_loss.item())


In [ ]:
test_df = pd.read_csv(f"{DATA_PATH}/test.csv")
test_df.head()


In [ ]:
X_test = []

for fname in tqdm(test_df["filename"]):
    file_path = f"{DATA_PATH}/{fname}"
    features = extract_features(file_path)
    X_test.append(features)

X_test = np.array(X_test)
print("X_test shape:", X_test.shape)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)


In [ ]:
model.eval()

with torch.no_grad():
    preds = model(X_test_tensor)
    preds = torch.argmax(preds, axis=1).cpu().numpy()

pred_labels = le.inverse_transform(preds)

submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": pred_labels
})

submission.to_csv("submission.csv", index=False)
submission.head()


# Milestone 1


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch
import warnings
warnings.filterwarnings("ignore")


DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)

DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
GENRES_ROOT = f"{DATA_ROOT}/genres_stems"

GENRES = sorted(os.listdir(GENRES_ROOT))
STEMS = {"drums.wav":"drums","vocals.wav":"vocals","bass.wav":"bass","other.wav":"other"}
STEM_KEYS = ['drums','vocals','bass','other']

GENRE_TO_TEST = 'rock'
SONG_INDEX = 0


def build_dataset(root_dir, val_split=0.17, seed=42):
    train_dataset = {g: {k: [] for k in STEM_KEYS} for g in GENRES}
    val_dataset   = {g: {k: [] for k in STEM_KEYS} for g in GENRES}
    
    rng = random.Random(seed)
    corrupted_count = 0
    small_count = 0
    large_count = 0
    
    for genre in GENRES:
        genre_path = os.path.join(root_dir,"genres_stems",genre)
        songs = sorted(os.listdir(genre_path))
        valid_songs = []
        
        for song in songs:
            song_path = os.path.join(genre_path,song)
            valid=True
            
            for stem_file in STEMS.keys():
                fp = os.path.join(song_path,stem_file)
                
                if not os.path.exists(fp):
                    valid=False
                else:
                    size=os.path.getsize(fp)
                    
                    if size < 4*1024:
                        corrupted_count+=1
                        valid=False
                    
                    size_mb=size/(1024*1024)
                    if size_mb < 5.0491: small_count+=1
                    if size_mb > 5.0493: large_count+=1
            
            if valid:
                valid_songs.append(song)
        
        rng.shuffle(valid_songs)
        split=int(len(valid_songs)*(1-val_split))
        train_songs=valid_songs[:split]
        val_songs=valid_songs[split:]
        
        for s in train_songs:
            for stem_file,stem_key in STEMS.items():
                train_dataset[genre][stem_key].append(
                    os.path.join(genre_path,s,stem_file))
        
        for s in val_songs:
            for stem_file,stem_key in STEMS.items():
                val_dataset[genre][stem_key].append(
                    os.path.join(genre_path,s,stem_file))
    
    Q1 = corrupted_count + small_count
    Q2 = abs(large_count - small_count)
    Q3 = abs(len(train_dataset["reggae"]["drums"]) - len(val_dataset["country"]["vocals"]))
    
    return train_dataset, val_dataset, Q1, Q2, Q3

tr, val, Q1, Q2, Q3 = build_dataset(DATA_ROOT)

In [ ]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    records = []

    for genre in dataset_dict:
        for stem in dataset_dict[genre]:
            for file_path in dataset_dict[genre][stem]:

                y, _ = librosa.load(file_path, sr=sr)

                total_duration = len(y) / sr

                rms = librosa.feature.rms(
                    y=y,
                    frame_length=N_FFT,
                    hop_length=HOP_LENGTH
                )[0]

                rms_db = librosa.amplitude_to_db(rms, ref=np.max)

                silent_frames = rms_db < -top_db

                silence_lengths = []
                count = 0

                for val in silent_frames:
                    if val:
                        count += 1
                    else:
                        if count > 0:
                            silence_lengths.append(count * HOP_LENGTH / sr)
                            count = 0

                if count > 0:
                    silence_lengths.append(count * HOP_LENGTH / sr)

                if len(silence_lengths) == 0:
                    continue

                max_silence = max(silence_lengths)

                if max_silence >= threshold_sec:

                    silence_type = []

                    if silence_lengths[0] >= threshold_sec:
                        silence_type.append("start")

                    if silence_lengths[-1] >= threshold_sec:
                        silence_type.append("end")

                    if max_silence >= threshold_sec and not silence_type:
                        silence_type.append("middle")

                    records.append({
                        "Genre": genre,
                        "Stem": stem,
                        "Duration": round(total_duration, 2),
                        "Max_Silence_Sec": round(max_silence, 2),
                        "Silence_Location": ", ".join(silence_type),
                        "File_Path": file_path
                    })

    columns = ["Genre","Stem","Duration","Max_Silence_Sec","Silence_Location","File_Path"]
    return pd.DataFrame(records, columns=columns)


In [ ]:
df_silence = find_long_silences(tr)


Q4=len(df_silence)
Q5=len(df_silence[df_silence["Stem"]=="vocals"])
Q6=df_silence[df_silence["Stem"]=="vocals"]["Max_Silence_Sec"].mean()
Q7=len(df_silence[(df_silence["Genre"]=="jazz") & (df_silence["Stem"]=="drums")])
Q8=len(df_silence[(df_silence["Genre"]=="jazz") & (df_silence["Stem"]=="drums") & (df_silence["Silence_Location"]=="middle")])
Q9=len(df_silence[(df_silence["Genre"]=="jazz") & (df_silence["Stem"]=="drums") & (df_silence["Max_Silence_Sec"]>=10)])


stems_audio = []
for key in STEM_KEYS:
    file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]
    y, _ = librosa.load(file_path, sr=SR, duration=DURATION)
    stems_audio.append(y)

stems_stack = np.vstack(stems_audio)

mix_raw = np.sum(stems_stack, axis=0)

rms_val = np.sqrt(np.mean(mix_raw ** 2))

peak_raw = np.max(np.abs(mix_raw))

mix_norm = mix_raw / peak_raw if peak_raw > 0 else mix_raw

Q10=len(mix_raw)
Q11=rms_val
Q12=np.max(np.abs(mix_raw))


print("\nFINAL ANSWERS:")
print("Q1:",Q1)
print("Q2:",Q2)
print("Q3:",Q3)
print("Q4:",Q4)
print("Q5:",Q5)
print("Q6:",Q6)
print("Q7:",Q7)
print("Q8:",Q8)
print("Q9:",Q9)
print("Q10:",Q10)
print("Q11:",Q11)
print("Q12:",Q12)